# 12 — Entry/Exit Engine & Investor Horizons

Phase 12 of the Indian Equity AI project plan.

Builds two horizon-specific models (swing/short-term and long-term) with different
feature emphasis, and a rule-based entry/exit decision-support layer on top of the
swing model's predictions.

**Intraday is out of scope**, honestly: this project only has daily EOD data (Phase 1).
Section 26's intraday feature list (1/5/15-minute bars, intraday VWAP) needs a
different, likely paid, real-time/historical intraday data source -- not something
to fabricate. "Do not use the same model for every investment horizon" (Section 26)
is satisfied by the two horizons built here using genuinely different data.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from lightgbm import LGBMClassifier

from src.models.dataset import FEATURE_COLUMNS, build_feature_table, chronological_split
from src.models.horizons import LONG_TERM_HORIZON_DAYS, SWING_HORIZON_DAYS, add_horizon_target
from src.models.multimodal_dataset import FUNDAMENTAL_FEATURE_COLUMNS, merge_fundamentals_pit
from src.models.evaluate import classification_metrics
from src.entryexit.engine import compute_entry_exit

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RANDOM_STATE = 42


def make_lgbm():
    return LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1,
    )

## Horizon 1: Swing / short-term (5-day forward direction, technical features)

In [2]:
technical = pd.read_parquet(DATA_PROCESSED / "technical_features.parquet")
swing_df = build_feature_table(technical)
swing_df = add_horizon_target(swing_df, horizon_days=SWING_HORIZON_DAYS, target_col="swing_target")
swing_dataset = swing_df.dropna(subset=FEATURE_COLUMNS + ["swing_target"]).reset_index(drop=True)

swing_train, swing_val, swing_test = chronological_split(swing_dataset, train_frac=0.7, val_frac=0.15)

swing_model = make_lgbm()
swing_model.fit(swing_train[FEATURE_COLUMNS], swing_train["swing_target"])

swing_test = swing_test.copy()
swing_test["probability_up"] = swing_model.predict_proba(swing_test[FEATURE_COLUMNS])[:, 1]
swing_pred = swing_model.predict(swing_test[FEATURE_COLUMNS])

swing_metrics = classification_metrics(
    swing_test["swing_target"].values, swing_pred, swing_test["probability_up"].values
)
print("Swing (5-day) rows:", len(swing_dataset))
swing_metrics

Swing (5-day) rows: 51799


{'accuracy': 0.4592896525195538,
 'precision': np.float64(0.44539276118223486),
 'recall': np.float64(0.7992058990357346),
 'f1': np.float64(0.5720085253222369),
 'roc_auc': np.float64(0.4777538366657158),
 'log_loss': 0.743947870439368,
 'brier_score': np.float64(0.27263429092192937)}

## Horizon 2: Long-term (60-day forward direction, fundamentals-driven)

In [3]:
fundamentals_annual = pd.read_parquet(DATA_PROCESSED / "fundamentals_annual.parquet")

long_term_base = technical[["date", "symbol", "close"]].copy()
long_term_df = merge_fundamentals_pit(long_term_base, fundamentals_annual)
long_term_df = add_horizon_target(long_term_df, horizon_days=LONG_TERM_HORIZON_DAYS, target_col="long_term_target")
long_term_dataset = long_term_df.dropna(
    subset=FUNDAMENTAL_FEATURE_COLUMNS + ["long_term_target"]
).reset_index(drop=True)

lt_train, lt_val, lt_test = chronological_split(long_term_dataset, train_frac=0.7, val_frac=0.15)

long_term_model = make_lgbm()
long_term_model.fit(lt_train[FUNDAMENTAL_FEATURE_COLUMNS], lt_train["long_term_target"])

lt_test = lt_test.copy()
lt_test["probability_up"] = long_term_model.predict_proba(lt_test[FUNDAMENTAL_FEATURE_COLUMNS])[:, 1]
lt_pred = long_term_model.predict(lt_test[FUNDAMENTAL_FEATURE_COLUMNS])

long_term_metrics = classification_metrics(
    lt_test["long_term_target"].values, lt_pred, lt_test["probability_up"].values
)
print("Long-term (60-day) rows:", len(long_term_dataset))
long_term_metrics

Long-term (60-day) rows: 25903


{'accuracy': 0.5514961150603406,
 'precision': np.float64(0.533181522446324),
 'recall': np.float64(0.5618786424408639),
 'f1': np.float64(0.5471540644299783),
 'roc_auc': np.float64(0.5704662762132056),
 'log_loss': 0.8932708820650649,
 'brier_score': np.float64(0.3048541435989094)}

## Compare horizons

In [4]:
horizon_comparison = pd.DataFrame({"swing_5d": swing_metrics, "long_term_60d": long_term_metrics}).T
horizon_comparison

,accuracy,precision,recall,f1,roc_auc,log_loss,brier_score
swing_5d,0.459290,0.445393,0.799206,0.572009,0.477754,0.743948,0.272634
long_term_60d,0.551496,0.533182,0.561879,0.547154,0.570466,0.893271,0.304854


## Entry/Exit engine, on the swing model's latest predictions per stock

"Potential Entry Zone / Stop Zone / Target Zone" (project plan Section 27) --
**hypothetical, model-generated levels, not guaranteed prices.**

In [5]:
entry_exit = compute_entry_exit(
    swing_test["close"], swing_test["atr_14"], swing_test["probability_up"],
    threshold=0.55, stop_atr_multiple=1.5, reward_risk_ratio=2.0,
)
swing_test = pd.concat([swing_test.reset_index(drop=True), entry_exit.reset_index(drop=True)], axis=1)

latest = swing_test.sort_values("date").groupby("symbol").tail(1)
for _, row in latest.iterrows():
    print(f"\n{row['symbol']} -- {row['date'].date()} | P(up over {SWING_HORIZON_DAYS}d) = {row['probability_up']:.1%}")
    if row["has_signal"]:
        print(f"  Potential Entry Zone   Rs{row['entry_low']:.2f} - Rs{row['entry_high']:.2f}")
        print(f"  Potential Stop Zone    Rs{row['stop']:.2f}")
        print(f"  Potential Target Zone  Rs{row['target']:.2f}")
    else:
        print("  No trade suggested (signal below threshold)")

print("\n(These are model-generated hypothetical levels, not guaranteed prices.)")


TMPV -- 2026-09-21 | P(up over 5d) = 50.2%
  No trade suggested (signal below threshold)

CIPLA -- 2026-09-21 | P(up over 5d) = 47.5%
  No trade suggested (signal below threshold)

TCS -- 2026-09-21 | P(up over 5d) = 63.2%
  Potential Entry Zone   Rs2128.70 - Rs2139.34
  Potential Stop Zone    Rs2030.79
  Potential Target Zone  Rs2324.52

ULTRACEMCO -- 2026-09-21 | P(up over 5d) = 54.8%
  No trade suggested (signal below threshold)

ICICIBANK -- 2026-09-21 | P(up over 5d) = 60.5%
  Potential Entry Zone   Rs1345.00 - Rs1351.72
  Potential Stop Zone    Rs1315.24
  Potential Target Zone  Rs1404.51

EICHERMOT -- 2026-09-21 | P(up over 5d) = 49.2%
  No trade suggested (signal below threshold)

HINDUNILVR -- 2026-09-21 | P(up over 5d) = 54.2%
  No trade suggested (signal below threshold)

TECHM -- 2026-09-21 | P(up over 5d) = 57.2%
  Potential Entry Zone   Rs1558.50 - Rs1566.29
  Potential Stop Zone    Rs1499.51
  Potential Target Zone  Rs1676.47

ADANIPORTS -- 2026-09-21 | P(up over 5d) = 

## Save

In [6]:
horizon_comparison.to_csv(DATA_PROCESSED / "horizon_comparison.csv")

entry_exit_output = swing_test[
    ["date", "symbol", "close", "probability_up", "has_signal", "entry_low", "entry_high", "stop", "target"]
]
entry_exit_output.to_parquet(DATA_PROCESSED / "entry_exit_output.parquet", index=False)

print("Saved:", DATA_PROCESSED / "horizon_comparison.csv")
print("Saved:", DATA_PROCESSED / "entry_exit_output.parquet")
print("Fraction of swing test rows with a suggested trade:", entry_exit_output["has_signal"].mean())

Saved: D:\Practise\Stock Market Analysis\data\processed\horizon_comparison.csv
Saved: D:\Practise\Stock Market Analysis\data\processed\entry_exit_output.parquet
Fraction of swing test rows with a suggested trade: 0.5632773432491345


## Reflection

- **Intraday** is genuinely out of scope for this project's data (daily EOD only) --
  documented rather than faked.
- **Swing (5-day) vs long-term (60-day):** compare the ROC-AUC/log-loss/Brier numbers
  above. Given Phases 5/7/8/9/10 consistently found next-day direction near base rate,
  it is genuinely useful information either way: if the long-term, fundamentals-driven
  model does noticeably better than the technical-driven swing model, that would support
  H2 (fundamentals matter more at longer horizons) in a way Phase 7's next-day test
  couldn't show; if it doesn't, that's honest evidence H2 needs richer fundamentals data
  (Phase 4/7's known coverage gaps) before it can be properly tested.
- **Entry/exit levels are a rules-based decision-support layer, not a new prediction** --
  they mechanically translate an existing probability + ATR into a zone/stop/target using
  a fixed, documented risk/reward convention (1.5x ATR stop, 2:1 reward:risk target).
  Changing those constants changes the zones without changing the underlying forecast's
  quality -- they are a presentation/risk-management layer on top of Phase 5-10's models,
  not a new source of edge.
- Consistent with every prior phase: these are hypothetical, model-generated levels for a
  research/decision-support tool, not guaranteed prices or investment advice.